# Challenge: Interval arithmetic

In [ ]:
#    APM41012EP course notebook - Chapter 6 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Challenge: Interval arithmetic
#    
#    Authors: M. Breden, L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from mpmath import iv
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "seaborn"

## Newton's method

In [ ]:
def newton(f, df, x0, tol=1e-14, nitmax=30):

    # initialisation
    x = np.zeros(nitmax+1)
    x[0] = x0

    # Newton iteration
    for i in range(1, nitmax+1):
        x[i] = x[i-1] - f(x[i-1])/df(x[i-1])
        print(f"it = {i:3d}, xn = {x[i]:16.9e}, |f(xn)| = {abs(f(x[i])):14.8e}, |xn - xn-1| = {abs(x[i-1]-x[i]):14.8e}" )
        if ( abs(f(x[i])) < tol ): break
 
    return x[0:i+1]

We consider the function:

$$
f(x) = x^2 + 10^{-15}
$$

This function is strictly positive.

In [ ]:
def f(x):
    return x**2 + 1.e-15

def df(x):
    return 2*x 

In [ ]:
xmin=-0.5; xmax=0.5
x = np.linspace(xmin, xmax, 1000)   
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f(x), name='f(x)', showlegend=True))
fig.add_shape(type="line", x0=xmin, y0=0, x1=xmax, y1=0)
fig.show()

In [ ]:
xsol = newton(f, df, x0=1, tol=1.e-10, nitmax=100)

**It is instructive to try several tolerances:** 

* $\epsilon = 1.e-10$ --> solution: 7.62935084e-06 (17 iterations)
* $\epsilon = 1.e-11$ --> solution: 1.90717387e-06 (19 iterations)
* $\epsilon = 1.e-12$ --> solution: 9.53324765e-07 (20 iterations)
* $\epsilon = 1.e-13$ --> solution: 2.37018835e-07 (22 iterations)
* $\epsilon = 1.e-14$ --> solution: 5.39044033e-08 (24 iterations)
* $\epsilon = 1.e-15$ --> no convergence after 100 iterations

## Interval arithmetic

While Newton's algorithm in double precision may run into some trouble here, a simple application of interval arithmetic makes it possible to conclude whether or not a solution exists. One may refer to the article by Taschini (2008), proposed in the additional bibliography, for an algorithm solving a nonlinear equation by a Newton method using interval arithmetic, or to the book by Warwick Tucker, *Validated numerics, A short introduction to rigorous computations*, Princeton University Press, 2011.

In [ ]:
X = iv.mpf([xmin,xmax])
print(f(X))

**Challenge:** propose an algorithm using interval arithmetic to find the solution of a nonlinear equation and apply it to the function $f(x)$.

### Interval Newton method

The interval Newton method makes it possible not only to prove the existence of zeros of $f$, but also to prove the absence of zeros in certain intervals.

**Theoretical study**

Given a representable interval $X$ and $\hat x$ the midpoint of $X$, we denote by $T(X)$ a representable interval containing

$$
\hat x - \frac{f(\hat x)}{f'(X)},
$$

where $f'(X)$ denotes a representable interval containing $\{ f'(x) \,|\, x\in X\}$.

One can then show that:
- if $f$ has zeros in $X$, then they are necessarily contained in $T(X)$;

- if $0\notin f'(X)$ and $T(X)\subset X$, then there exists a unique zero of $f$ in $X$. 

**Algorithm**

Given a representable interval $X_0$, the interval Newton method consists of a sequence of representable intervals $X_k$ defined by:

$$
X_{k+1} = T(X_k) \cap X_k.
$$

In practice, if $0\in f'(X_k) = [\alpha_k,\beta_k]$, we take for $T(X_k)$ two representable intervals whose union contains $\displaystyle \hat x - \frac{f(\hat x)}{f'(X)}$, that is to say:

$$\hat x - \frac{f(\hat x)}{[\alpha_k,0]} \qquad{}\text{and}\qquad{}\hat x - \frac{f(\hat x)}{[0,\beta_k]},
$$

and we carry on the iterations on these two intervals independently. Thus, at iteration $k$, $X_k$ is in fact not necessarily a representable interval, but rather a finite union of representable intervals, on which we iterate independently in order to obtain $X_{k+1}$. By construction, $X_k$ necessarily contains all the zeros of $f$ belonging to $X_0$.

In [ ]:
def intersect(X, Y):
    """
    Computes the intersection of two intervals
    ----------   
    parameters:
    X, Y : intervals 
    
    return values:
    Z : X inter Y
    """
    a = max(X.a,Y.a)
    b = min(X.b,Y.b)
    if a <= b:
        Z = iv.mpf([a,b])
    else:
        Z = []        
    return Z

def interval_Newton(f, df ,X ,it_max=100, tol=10**-14):
    """
    Interval Newton method
    ----------   
    parameters:
    f : the function f (must be able to handle interval arguments)
    df : the function f' (must be able to handle interval arguments)
    X : the starting interval
    it_max : the maximum number of iterations
    tol : the tolerance
    
    return values:
    A list of intervals, which contain all the zeros of f in X
    """
    it = 0
    while X.delta > tol and it < it_max: #We stop if the size of the interval is smaller than tol
        xhat = X.mid
        dfX = df(X)
        if not(0 in dfX): #If f'(X) does not contain 0, we perform a "normal" iteration
            Xold = X
            X = xhat - f(xhat)/dfX
            X = intersect(X, Xold)
            it = it + 1
            if X == Xold or X == []: #If we get the same interval as at the start, or the empty set, we stop
                break
        else: #If f'(X) contains 0, we split T(X) into two disjoint intervals
            if 0 in f(xhat): #Degenerate case where we are potentially close to a multiple root
                break
            X1 = xhat - f(xhat)/iv.mpf([dfX.a,0]) # first interval
            X1 = intersect(X1, X)
            if not(X1 == []):
                X1 = interval_Newton(f, df ,X1 ,it_max-(it+1), tol)
            
            X2 = xhat - f(xhat)/iv.mpf([0,dfX.b]) #second interval
            X2 = intersect(X2, X)
            if not(X2 == []):
                X2 = interval_Newton(f, df ,X2 ,it_max-(it+1), tol)
            
            X = [*X1, *X2]
            break
    if not(type(X)==list):
        X = [X]

    return X

**Test on the function $g(x) = x^3 - 2x -5$**

In [ ]:
def g(x):
    return x**3 - 2*x - 5

def dg(x):
    return 3*x**2 - 2

In [ ]:
xmin=-2; xmax=3
x = np.linspace(xmin, xmax, 1000)   
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=g(x), name='g(x)', showlegend=True))
fig.show()

In [ ]:
X = iv.mpf([xmin,xmax])
Xsol = interval_Newton(g, dg, X)
iv.pretty = True
print(Xsol)

**Test on the function $f(x) = x^2 + 10^{-15}$**

In [ ]:
xmin=-5; xmax=5
X = iv.mpf([xmin,xmax])
Xsol = interval_Newton(f, df, X)
iv.pretty = True
print(Xsol)

**Test on the function $\displaystyle h(x) = \exp(\sin(20x)) + x^4 +\frac{x}{1+x^2} - 2$**

In [ ]:
def h(x):
    return np.exp(np.sin(20*x)) + x**4 + x/(1.+x**2) - 2

def ivh(x):
    return iv.exp(iv.sin(20*x)) + x**4 + x/(1.+x**2) - 2

def ivdh(x):
    return 20*iv.cos(20*x)*iv.exp(iv.sin(20*x)) + 4*x**3 + (1-x**2)/(1+x**2)**2

In [ ]:
xmin=-2; xmax=2
x = np.linspace(xmin, xmax, 1000)   
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=h(x), name='h(x)', showlegend=True))
fig.show()

In [ ]:
xmin=-2; xmax=2
X = iv.mpf([xmin,xmax])
Xsol = interval_Newton(ivh, ivdh, X)
print(f'The zeros of g in {X} are necessarily contained in the following intervals:\n')
for k in range(len(Xsol)):
    print(Xsol[k])